# 76 — Delta-ML: Template-Based Δ-pEC50 Prediction

**The RNA Analogy applied to chemistry:**
Instead of predicting absolute pEC50, find the closest training compound (the *template*),
then predict only the **delta** (how much the test compound differs in activity).

Train the delta model on all ~800k+ training pairs with Tanimoto > 0.35.
At inference: pEC50_test ≈ pEC50_template + Δ_model(diff_features).

This directly models activity cliffs as large Δ values — exactly where standard
models fail most. Works best precisely when test ≈ train (our situation: median sim=0.52).


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
# ── Build training pairs ──────────────────────────────────────────────────────
# Tanimoto from Morgan FPs (binary, so dot/union is exact Tanimoto)
print("Computing pairwise Tanimoto for training set...", flush=True)
dot = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union = rowsum[:,None] + rowsum[None,:] - dot
tanimoto = np.where(union>0, dot/union, 0.0)
np.fill_diagonal(tanimoto, 0)

SIM_THRESH = 0.35   # minimum similarity to form a pair
MAX_PAIRS   = 300_000  # cap to avoid memory blow-up

# Physchem diffs for delta features
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)

print("Building pair dataset...", flush=True)
rng = np.random.default_rng(42)
i_idx, j_idx = np.where(tanimoto >= SIM_THRESH)
# Remove symmetric duplicates (keep only i < j)
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Total pairs with Tanimoto≥{SIM_THRESH}: {len(i_idx):,}")

if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
    print(f"Downsampled to {MAX_PAIRS:,} pairs")

# Delta target: pEC50_j - pEC50_i  (also build reverse pairs for symmetry)
y_delta_ij = y_tr[j_idx] - y_tr[i_idx]
y_delta_ji = -y_delta_ij

# Features: [common_fp, diff_fp (XOR), sim, anchor_pec50, delta_physchem]
fps_i = fps_tr[i_idx]; fps_j = fps_tr[j_idx]
fp_common = np.minimum(fps_i, fps_j)             # AND (common substructure)
fp_diff   = np.abs(fps_i - fps_j).astype(np.float32)  # XOR proxy
sims = tanimoto[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
phys_diff_ji = -phys_diff_ij

# Build final feature matrices (ij and ji directions)
def make_delta_feats(fp_anc, fp_common, fp_diff, sim, anc_pec50, phys_diff):
    # 64-dim fingerprint compression to keep features manageable
    fp_common_64 = fp_common.reshape(-1, 32, fp_common.shape[1]//32).mean(-1) if fp_common.shape[1]>64 else fp_common
    fp_diff_64   = fp_diff.reshape(-1, 32, fp_diff.shape[1]//32).mean(-1)   if fp_diff.shape[1]>64 else fp_diff
    return np.hstack([fp_common_64, fp_diff_64, sim,
                      anc_pec50[:,None], phys_diff])

print("Building feature matrices...", flush=True)
F_ij = make_delta_feats(fps_i, fp_common, fp_diff, sims,
                         y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_j, fp_common, fp_diff, sims,
                         y_tr[j_idx], phys_diff_ji)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_delta_ij, y_delta_ji])
print(f"Delta dataset: {F_all.shape}  Δ range [{y_all.min():.2f}, {y_all.max():.2f}]")


Computing pairwise Tanimoto for training set...


Computing physchem...


Building pair dataset...


Total pairs with Tanimoto≥0.35: 5,186
Building feature matrices...


Delta dataset: (10372, 73)  Δ range [-4.68, 4.68]


In [5]:
# ── Train delta model ──────────────────────────────────────────────────────────
from sklearn.model_selection import cross_val_score
print("Training delta model on all pairs...", flush=True)
delta_model = lgb.LGBMRegressor(**{**LGBM, "n_estimators": 500})
delta_model.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("Delta model trained.")

# Sanity check: predict delta for same-compound (should be ~0)
F_self = make_delta_feats(fps_tr[:10], np.minimum(fps_tr[:10],fps_tr[:10]),
                           np.zeros((10,fps_tr.shape[1]),dtype=np.float32),
                           np.ones((10,1)),y_tr[:10], np.zeros((10,len(props)),dtype=np.float32))
self_deltas = delta_model.predict(F_self)
print(f"Self-delta (should be ~0): mean={self_deltas.mean():.3f} std={self_deltas.std():.3f}")

Training delta model on all pairs...


Delta model trained.
Self-delta (should be ~0): mean=-0.252 std=0.284


In [6]:
# ── Scaffold CV: evaluate delta-ML vs direct prediction ────────────────────────
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_delta = np.full(len(y_tr), np.nan)
oof_direct = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct model (baseline within fold)
    m_direct = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                         valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                         callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_direct.predict(X_tr[va_idx])

    # Delta-ML: for each val compound, find nearest TRAINING neighbor
    fps_val = fps_tr[va_idx]; fps_fold_tr = fps_tr[tr_idx]
    dot_vt = (fps_val @ fps_fold_tr.T).astype(np.float32)
    rs_v = fps_val.sum(1)[:,None]; rs_t = fps_fold_tr.sum(1)[None,:]
    sim_vt = dot_vt / np.maximum(rs_v + rs_t - dot_vt, 1e-6)
    best_t_idx = sim_vt.argmax(1)   # index into fold-train compounds
    best_sims = sim_vt.max(1)
    best_global = tr_idx[best_t_idx]  # global indices

    phys_val = phys_arr[va_idx]; phys_refs = phys_arr[best_global]
    fp_refs   = fps_tr[best_global]; fp_val = fps_val
    fp_com    = np.minimum(fp_val, fp_refs)
    fp_dif    = np.abs(fp_val - fp_refs).astype(np.float32)
    F_test    = make_delta_feats(fp_refs, fp_com, fp_dif, best_sims[:,None],
                                  y_tr[best_global], phys_val - phys_refs)
    delta_pred = delta_model.predict(F_test)
    oof_delta[va_idx] = y_tr[best_global] + delta_pred

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_del = rae(y_tr[va_idx], oof_delta[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  delta={r_del:.4f}  "
          f"best_sim_mean={best_sims.mean():.3f}", flush=True)

m_dir  = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_del  = full_metrics(y_tr, oof_delta,  cliff_pairs, "delta_ml")

# Blend: alpha * delta + (1-alpha) * direct, sweep alpha
best_alpha, best_rae = 0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.1, 1.01, 0.1):
    blended = alpha * oof_delta + (1-alpha) * oof_direct
    r = rae(y_tr[np.isfinite(blended)], blended[np.isfinite(blended)])
    if r < best_rae:
        best_rae, best_alpha = r, alpha

oof = best_alpha * oof_delta + (1-best_alpha) * oof_direct
m_blend = full_metrics(y_tr, oof, cliff_pairs, f"blend(α={best_alpha:.1f})")
print(f"\nBest blend α={best_alpha:.1f}  OOF RAE={best_rae:.4f}")
results_df = pd.DataFrame([m_dir, m_del, m_blend],
                           index=["direct","delta_ml",f"blend_{best_alpha:.1f}"])
print("\n" + results_df.round(4).to_string())



=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  delta=0.3781  best_sim_mean=0.394


  fold 2  direct=0.5759  delta=0.4034  best_sim_mean=0.391


  fold 3  direct=0.6021  delta=0.4413  best_sim_mean=0.387


  fold 4  direct=0.5665  delta=0.4369  best_sim_mean=0.388


  fold 5  direct=0.6033  delta=0.4393  best_sim_mean=0.390


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  [delta_ml] RAE=0.4164 MAE=0.3789 R²=0.7334 r=0.8603 ρ=0.8093 τ=0.6449  Cliff=nan
  [blend(α=1.0)] RAE=0.4164 MAE=0.3789 R²=0.7334 r=0.8603 ρ=0.8093 τ=0.6449  Cliff=nan

Best blend α=1.0  OOF RAE=0.4164

              RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
direct     0.5643  0.5134  0.5991   0.7740    0.7268   0.5345        NaN
delta_ml   0.4164  0.3789  0.7334   0.8603    0.8093   0.6449        NaN
blend_1.0  0.4164  0.3789  0.7334   0.8603    0.8093   0.6449        NaN


In [7]:
# ── Final predictions ─────────────────────────────────────────────────────────
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

# Delta for test compounds: nearest training neighbor
dot_tt = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tt / np.maximum(rs_te + rs_tr_v - dot_tt, 1e-6)
best_tr_idx = sim_te_tr.argmax(1); best_sims_te = sim_te_tr.max(1)
print(f"Test→train similarity: mean={best_sims_te.mean():.3f}  "
      f"min={best_sims_te.min():.3f}  max={best_sims_te.max():.3f}")

phys_te_arr = np.array([[p.get(k,0) or 0 for k in ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]]
                          for p in te["smiles"].map(compute_physchem)], dtype=np.float32)
fp_refs_te   = fps_tr[best_tr_idx]
fp_com_te    = np.minimum(fps_te, fp_refs_te)
fp_dif_te    = np.abs(fps_te - fp_refs_te).astype(np.float32)
F_te_delta   = make_delta_feats(fp_refs_te, fp_com_te, fp_dif_te,
                                  best_sims_te[:,None], y_tr[best_tr_idx],
                                  phys_te_arr - phys_arr[best_tr_idx])
te_delta_pred = delta_model.predict(F_te_delta)
te_delta_final = y_tr[best_tr_idx] + te_delta_pred

te_preds = best_alpha*te_delta_final + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_delta_ml.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_ml.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"76_delta_ml_template.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Test→train similarity: mean=0.532  min=0.323  max=0.806


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\76_delta_ml_template.csv
Test: min=2.91 med=4.88 max=5.91
